# Clase 009 — Excepciones y context managers

**Parte 0** · Python Tutorial cap. 8 + Ramalho cap. 18.

> 🎯 Manejo riguroso de errores y garantía de cleanup con `with`.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import os, time, tempfile
from contextlib import contextmanager
from pathlib import Path

## 1️⃣ Jerarquía de excepciones

```
BaseException
├── SystemExit          ← sys.exit()
├── KeyboardInterrupt   ← Ctrl+C
└── Exception           ← captura esto, no BaseException
    ├── ArithmeticError
    │   └── ZeroDivisionError
    ├── LookupError
    │   ├── KeyError
    │   └── IndexError
    ├── ValueError
    ├── TypeError
    ├── OSError
    │   └── FileNotFoundError
    └── ...
```

Captura `Exception` o más específico. **NUNCA** captures `BaseException` o uses `except:` solo.

## 2️⃣ `try/except/else/finally`

```python
try:
    valor = riesgoso()
except ValueError as e:
    log(f'valor inválido: {e}')
    valor = None
else:
    # Solo si try NO lanzó excepción
    log('OK')
finally:
    # Siempre, lanzó o no
    cleanup()
```

- `try` — código que puede fallar
- `except` — manejo específico
- `else` — éxito (raro, pero útil)
- `finally` — cleanup garantizado (lo que hace `with` automático)

In [ ]:
def parse_int_safe(s, default=0):
    """Convierte a int; default si no es parseable. Otros errores propagan."""
    try:
        return int(s)
    except ValueError:
        return default

print(parse_int_safe('42'))          # 42
print(parse_int_safe('foo'))         # 0
print(parse_int_safe('3.14'))        # 0
try:
    parse_int_safe({'a': 1})         # TypeError no es ValueError → propaga
except TypeError as e:
    print(f'TypeError correcto: {e}')

## 3️⃣ Capturar específico — por qué

```python
# ❌ TRAMPA: esconde TODO error, hasta tipo y nombre
try:
    valor = parse(linea)
except:
    valor = None   # bug silencioso

# ✅ CORRECTO: solo el error que sabes manejar
try:
    valor = parse(linea)
except ValueError as e:
    log(f'línea inválida {idx}: {e}')
    valor = None
```

Un `except:` puede ocultar un `KeyboardInterrupt`, un `NameError` (typo) o un `MemoryError`. Casi nunca es lo que quieres.

## 4️⃣ Excepciones propias

Las excepciones son **comunicación tipada**. En vez de:

```python
raise Exception('CSV corrupto en línea 42')
```

Define tu tipo:

```python
class DatasetCorruptoError(Exception):
    def __init__(self, mensaje, linea):
        super().__init__(mensaje)
        self.linea = linea

try:
    cargar(path)
except DatasetCorruptoError as e:
    log(f'línea {e.linea}: {e}')   # ahora caller puede ACTUAR
```

In [ ]:
class DatasetCorruptoError(Exception):
    def __init__(self, mensaje, linea):
        super().__init__(mensaje)
        self.linea = linea

def cargar_csv_estricto(lineas, n_cols):
    for i, linea in enumerate(lineas, start=1):
        cols = linea.split(',')
        if len(cols) != n_cols:
            raise DatasetCorruptoError(f'esperaba {n_cols} cols, vino {len(cols)}', linea=i)
        yield cols

datos = ['a,b,c', 'd,e,f', 'g,h']   # última línea corrupta
try:
    list(cargar_csv_estricto(datos, n_cols=3))
except DatasetCorruptoError as e:
    print(f'Error línea {e.linea}: {e}')

## 5️⃣ Context managers — `with`

```python
# Sin with: si parse() falla, el archivo queda abierto
f = open('data.csv')
datos = parse(f)
f.close()

# Con with: cleanup garantizado, incluso si parse() lanza
with open('data.csv') as f:
    datos = parse(f)
# aquí f ya está cerrado
```

Protocolo: el objeto debe tener `__enter__` (entrada) y `__exit__` (salida). `__exit__` recibe info de la excepción si la hubo.

In [ ]:
# Demo: with garantiza close incluso con excepción
tmp = Path(tempfile.mkdtemp()) / 'demo.txt'
tmp.write_text('linea1\nlinea2\nlinea3\n')

with open(tmp) as f:
    for linea in f:
        print(linea.strip())
print('archivo cerrado:', f.closed)

## 6️⃣ Context manager propio con `@contextmanager`

La forma corta: una función con `yield`. Antes del yield = `__enter__`. Después = `__exit__`.

```python
from contextlib import contextmanager

@contextmanager
def timer(label):
    t0 = time.perf_counter()
    yield                       # aquí corre el código del `with`
    dt = time.perf_counter() - t0
    print(f'{label}: {dt*1000:.1f} ms')

with timer('carga'):
    time.sleep(0.1)
```

In [ ]:
@contextmanager
def timer(label):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        print(f'{label}: {dt*1000:.1f} ms')

with timer('operación A'):
    time.sleep(0.05)

with timer('operación B'):
    sum(i*i for i in range(100_000))

In [ ]:
# Context manager práctico: cambiar de directorio temporalmente
@contextmanager
def cd(path):
    prev = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(prev)   # garantizado incluso si hay excepción

print('antes:', Path.cwd().name)
with cd(tempfile.gettempdir()):
    print('dentro:', Path.cwd().name)
print('después:', Path.cwd().name)

## ✅ Checklist

- [ ] Capturo excepciones específicas, no `except:`
- [ ] Sé crear una excepción propia con atributos
- [ ] Uso `with` para archivos y otros recursos
- [ ] Sé escribir un context manager con `@contextmanager`
- [ ] Entiendo que `finally` garantiza cleanup

## 📝 Homework

Ver `README.md`. `parse_int_safe`, `DatasetCorruptoError`, `timer`, `cd` context manager.

## 🔗 Referencias

- [Python Tutorial — Errors](https://docs.python.org/3/tutorial/errors.html)
- [contextlib](https://docs.python.org/3/library/contextlib.html)
- Ramalho, *Fluent Python* 2e, cap. 18

➡️ **Siguiente:** [010 — OOP básico, dataclasses, herencia](../010-oop-basico-dataclasses-herencia/README.md)